# All of Us Internal Models

Train and evaluate XGBoost models for pregnancy complication prediction using the All of Us cohort as both training and test data (internal validation)

**Models trained:**
- Pre-eclampsia (PE) at 12 weeks
- Gestational Diabetes (GDM) at 12 weeks
- Gestational Hypertension (GHTN) at 12 weeks

First half of file is commented out because it is dependent on the All of Us platform. The second half (modeling) uses synthetic data. 

# Setup 

In [1]:
import pandas as pd
import numpy as np
import os
import pickle
from scipy import stats

from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, roc_auc_score, 
                            average_precision_score, balanced_accuracy_score,
                            precision_score, recall_score, brier_score_loss, 
                            log_loss, roc_curve, auc, precision_recall_curve)
from sklearn.preprocessing import MinMaxScaler

from xgboost import XGBClassifier

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
'''
version = %env WORKSPACE_CDR
my_bucket = os.getenv('WORKSPACE_BUCKET')
'''

"\nversion = %env WORKSPACE_CDR\nmy_bucket = os.getenv('WORKSPACE_BUCKET')\n"

# Data Loading and Merging

In [3]:
'''
print("LOADING DATA FILES")

# Load all data components
df_outcomes = pd.read_pickle('pregnancy_cohort_outcomes_date.pkl')
df_hx = pd.read_pickle('pregnancy_cohort_hx_date.pkl')
df_demo = pd.read_csv('pregnancy_demographics_date.csv')
df_labs = pd.read_csv('labs_hold_24w_date.csv')
df_vitals = pd.read_csv('vitals_hold_24w_date.csv')

print(f" Outcomes: {df_outcomes.shape}")
print(f" Medical history: {df_hx.shape}")
print(f" Demographics: {df_demo.shape}")
print(f" Labs: {df_labs.shape}")
print(f" Vitals: {df_vitals.shape}")
'''

'\nprint("LOADING DATA FILES")\n\n# Load all data components\ndf_outcomes = pd.read_pickle(\'pregnancy_cohort_outcomes_6-18-25.pkl\')\ndf_hx = pd.read_pickle(\'pregnancy_cohort_hx_6-18-25.pkl\')\ndf_demo = pd.read_csv(\'pregnancy_demographics_5-29-25.csv\')\ndf_labs = pd.read_csv(\'labs_hold_24w_3-1-25.csv\')\ndf_vitals = pd.read_csv(\'vitals_hold_24w_5-29-25.csv\')\n\nprint(f" Outcomes: {df_outcomes.shape}")\nprint(f" Medical history: {df_hx.shape}")\nprint(f" Demographics: {df_demo.shape}")\nprint(f" Labs: {df_labs.shape}")\nprint(f" Vitals: {df_vitals.shape}")\n'

In [4]:
'''
# Merge all datasets
df = df_outcomes.merge(df_hx, how='left', on='id')
labs_vitals = df_vitals.merge(df_labs, how='outer', on='id', suffixes=[None, "_x"])
df = df.merge(labs_vitals, how='inner', on='id')
df = df.merge(df_demo, how='left', on='id')

print(f"\n Merged dataset: {df.shape}")
print(f"  Columns: {len(df.columns)}")
'''

'\n# Merge all datasets\ndf = df_outcomes.merge(df_hx, how=\'left\', on=\'id\')\nlabs_vitals = df_vitals.merge(df_labs, how=\'outer\', on=\'id\', suffixes=[None, "_x"])\ndf = df.merge(labs_vitals, how=\'inner\', on=\'id\')\ndf = df.merge(df_demo, how=\'left\', on=\'id\')\n\nprint(f"\n Merged dataset: {df.shape}")\nprint(f"  Columns: {len(df.columns)}")\n'

# Feature Eng for 12-Week Prediction

In [5]:
'''
# Remove late pregnancy (lp_) features for 12-week models
# Keep only pre-pregnancy (pp_) and early pregnancy (ep_) features
pp_ep = ~df.columns.str.contains("lp")
df = df.loc[:, pp_ep]

print(f" Features filtered to pre-pregnancy and early pregnancy only")
print(f" Dataset shape after filtering: {df.shape}")
'''

'\n# Remove late pregnancy (lp_) features for 12-week models\n# Keep only pre-pregnancy (pp_) and early pregnancy (ep_) features\npp_ep = ~df.columns.str.contains("lp")\ndf = df.loc[:, pp_ep]\n\nprint(f" Features filtered to pre-pregnancy and early pregnancy only")\nprint(f" Dataset shape after filtering: {df.shape}")\n'

In [6]:
'''
# Convert censor_date to datetime (remove timezone)
for i in range(len(df)):
    df.loc[i, 'censor_date_x'] = pd.to_datetime(df.loc[i, 'censor_date_x']).replace(tzinfo=None)

# Calculate maternal age at conception
df['age'] = (np.floor((pd.to_datetime(df['censor_date_x']) - pd.to_datetime(df['birth_date'])).dt.days / 365)).astype(int)

# Rename columns for clarity
df.rename(columns={
    'age': 'mage', 
    'pcos_hx': 'pcos', 
    'depression_hx': 'dep_priorlmp', 
    'pe_hx': 'hxpe_e',
    'Gestational Diabetes': 'GDM', 
    'Pre-eclampsia': 'PE', 
    'Gestational_Hypertension': 'GHTN',
    'Eclampsia': 'E'
}, inplace=True)

# Calculate parity (whether patient has had previous pregnancies)
df['parity'] = (df.groupby('person_id_x')['censor_date_x'].transform('nunique') > 1).astype(int)

# Create race/ethnicity categories
df['Hispanic'] = df['ethnicity'] == 'Hispanic or Latino'
df['NH_White'] = (df['ethnicity'] != 'Hispanic or Latino') & (df['race'] == 'White')
df['NH_Black'] = (df['ethnicity'] != 'Hispanic or Latino') & (df['race'] == 'Black or African American')
df['NH_Asian'] = (df['ethnicity'] != 'Hispanic or Latino') & (df['race'].isin(['Asian', 'Native Hawaiian or Other Pacific Islander']))
df['Other'] = (df['ethnicity'] != 'Hispanic or Latino') & (~df['race'].isin(['White', 'Black or African American', 'Asian', 'Native Hawaiian or Other Pacific Islander']))

df[['Hispanic', 'NH_White', 'NH_Black', 'NH_Asian', 'Other']] = df[['Hispanic', 'NH_White', 'NH_Black', 'NH_Asian', 'Other']].astype(int)

# Remove rows with no vital/lab data at all
df = df.dropna(subset=df.filter(regex=r"^(ep|pp)").columns, how="all")

print(f" Maternal age calculated")
print(f" Parity calculated")
print(f" Race/ethnicity categories created")
print(f" Rows without any vital/lab data removed")
print(f"\nFinal dataset shape: {df.shape}")
'''

'\n# Convert censor_date to datetime (remove timezone)\nfor i in range(len(df)):\n    df.loc[i, \'censor_date_x\'] = pd.to_datetime(df.loc[i, \'censor_date_x\']).replace(tzinfo=None)\n\n# Calculate maternal age at conception\ndf[\'age\'] = (np.floor((pd.to_datetime(df[\'censor_date_x\']) - pd.to_datetime(df[\'birth_date\'])).dt.days / 365)).astype(int)\n\n# Rename columns for clarity\ndf.rename(columns={\n    \'age\': \'mage\', \n    \'pcos_hx\': \'pcos\', \n    \'depression_hx\': \'dep_priorlmp\', \n    \'pe_hx\': \'hxpe_e\',\n    \'Gestational Diabetes\': \'GDM\', \n    \'Pre-eclampsia\': \'PE\', \n    \'Gestational_Hypertension\': \'GHTN\',\n    \'Eclampsia\': \'E\'\n}, inplace=True)\n\n# Calculate parity (whether patient has had previous pregnancies)\ndf[\'parity\'] = (df.groupby(\'person_id_x\')[\'censor_date_x\'].transform(\'nunique\') > 1).astype(int)\n\n# Create race/ethnicity categories\ndf[\'Hispanic\'] = df[\'ethnicity\'] == \'Hispanic or Latino\'\ndf[\'NH_White\'] = (df[\'e

In [7]:
'''
print(f"\nSample preview:")
print(df.head())
'''

'\nprint(f"\nSample preview:")\nprint(df.head())\n'

## Data Quality Check

In [8]:
'''
# Check data completeness
non_nan_percentage = (df.count() / len(df)) * 100
print(f"\nData completeness by column (showing extremes):")
print(f"Most complete:\n{non_nan_percentage.nlargest(5)}")
print(f"\nLeast complete:\n{non_nan_percentage.nsmallest(5)}")

# Clean BMI outliers (must be between 10 and 100)
df['PPREG_BMI'] = df['PPREG_BMI'].mask((df['PPREG_BMI'] < 10) | (df['PPREG_BMI'] > 100), np.nan)
print(f"\n BMI outliers (<10 or >100) set to NaN")
'''

'\n# Check data completeness\nnon_nan_percentage = (df.count() / len(df)) * 100\nprint(f"\nData completeness by column (showing extremes):")\nprint(f"Most complete:\n{non_nan_percentage.nlargest(5)}")\nprint(f"\nLeast complete:\n{non_nan_percentage.nsmallest(5)}")\n\n# Clean BMI outliers (must be between 10 and 100)\ndf[\'PPREG_BMI\'] = df[\'PPREG_BMI\'].mask((df[\'PPREG_BMI\'] < 10) | (df[\'PPREG_BMI\'] > 100), np.nan)\nprint(f"\n BMI outliers (<10 or >100) set to NaN")\n'

# Outcome Selection


Change this variable to train different models:
- `'Pre-eclampsia_12'` for Pre-eclampsia
- `'Gestational_Hypertension_12'` for Gestational Hypertension
- `'Gestational Diabetes_12'` for Gestational Diabetes

The feature list and data filtering will automatically adjust based on selection.

In [9]:
'''
outcome = 'Placental_Abruption'#'Gestational_Hypertension_12'  # Change this to 'Gestational_Hypertension_12' or 'Gestational Diabetes_12' as needed
print(f"OUTCOME SELECTED: {outcome}")
'''


'\noutcome = \'Placental_Abruption\'#\'Gestational_Hypertension_12\'  # Change this to \'Gestational_Hypertension_12\' or \'Gestational Diabetes_12\' as needed\nprint(f"OUTCOME SELECTED: {outcome}")\n'

# Feature Selection

Feature selection:
1. Start with all vitals and labs features (pp_, ep_, lp_)
2. Add medical history features
3. Add race/ethnicity categories
4. Conditionally add features based on outcome:
   - `dm` (diabetes): Added for all outcomes EXCEPT Gestational Diabetes
   - `chtn_prepreg` (chronic hypertension): Added for all outcomes EXCEPT Pre-eclampsia and Gestational Hypertension

In [10]:
'''
non_nan_percentage = (df.count() / len(df)) * 100

with pd.option_context('display.max_rows', None):
    print(non_nan_percentage)
'''

"\nnon_nan_percentage = (df.count() / len(df)) * 100\n\nwith pd.option_context('display.max_rows', None):\n    print(non_nan_percentage)\n"

In [11]:
'''
# Base features: ALL vitals and labs (pp_, ep_, lp_)
# Note: lp_ features will be removed for 12-week models, but we include them here
# so the logic works for models at different time points


feats_to_train = ['pp_dbp_avg', 'pp_hr_avg', 'pp_sbp_avg', 'pp_dbp_min', 'pp_hr_min', 'pp_sbp_min', 
                  'pp_dbp_max', 'pp_hr_max', 'pp_sbp_max', 'pp_dbp_std', 'pp_hr_std', 'pp_sbp_std', 
                  'ep_dbp_avg', 'ep_hr_avg', 'ep_sbp_avg', 'ep_dbp_min', 'ep_hr_min', 'ep_sbp_min', 
                  'ep_dbp_max', 'ep_hr_max', 'ep_sbp_max', 'ep_dbp_std', 'ep_hr_std', 'ep_sbp_std', 
                  'lp_dbp_avg', 'lp_hr_avg', 'lp_sbp_avg', 'lp_dbp_min', 'lp_hr_min', 'lp_sbp_min', 
                  'lp_dbp_max', 'lp_hr_max', 'lp_sbp_max', 'lp_dbp_std', 'lp_hr_std', 'lp_sbp_std', 
                  'pp_ALT_avg', 'pp_ANC_avg', 'pp_AST_avg', 'pp_BUN_avg', 'pp_CALCIUM_avg', 'pp_CHLORIDE_avg', 
                  'pp_CREATININE_avg', 'pp_GLU_RAN_avg', 'pp_HGB_avg', 'pp_RBC_avg', 'pp_WBC_avg', 
                  'pp_ALT_min', 'pp_ANC_min', 'pp_AST_min', 'pp_BUN_min', 'pp_CALCIUM_min', 'pp_CHLORIDE_min', 
                  'pp_CREATININE_min', 'pp_GLU_RAN_min', 'pp_HGB_min', 'pp_RBC_min', 'pp_WBC_min', 
                  'pp_ALT_max', 'pp_ANC_max', 'pp_AST_max', 'pp_BUN_max', 'pp_CALCIUM_max', 'pp_CHLORIDE_max', 
                  'pp_CREATININE_max', 'pp_GLU_RAN_max', 'pp_HGB_max', 'pp_RBC_max', 'pp_WBC_max', 
                  'pp_ALT_std', 'pp_ANC_std', 'pp_AST_std', 'pp_BUN_std', 'pp_CALCIUM_std', 'pp_CHLORIDE_std', 
                  'pp_CREATININE_std', 'pp_GLU_RAN_std', 'pp_HGB_std', 'pp_RBC_std', 'pp_WBC_std', 
                  'ep_ALT_avg', 'ep_ANC_avg', 'ep_AST_avg', 'ep_BUN_avg', 'ep_CALCIUM_avg', 'ep_CHLORIDE_avg', 
                  'ep_CREATININE_avg', 'ep_GLU_RAN_avg', 'ep_HGB_avg', 'ep_RBC_avg', 'ep_WBC_avg', 
                  'ep_ALT_min', 'ep_ANC_min', 'ep_AST_min', 'ep_BUN_min', 'ep_CALCIUM_min', 'ep_CHLORIDE_min', 
                  'ep_CREATININE_min', 'ep_GLU_RAN_min', 'ep_HGB_min', 'ep_RBC_min', 'ep_WBC_min', 
                  'ep_ALT_max', 'ep_ANC_max', 'ep_AST_max', 'ep_BUN_max', 'ep_CALCIUM_max', 'ep_CHLORIDE_max', 
                  'ep_CREATININE_max', 'ep_GLU_RAN_avg', 'ep_HGB_max', 'ep_RBC_max', 'ep_WBC_max', 
                  'ep_ALT_std', 'ep_ANC_std', 'ep_AST_std', 'ep_BUN_std', 'ep_CALCIUM_std', 'ep_CHLORIDE_std', 
                  'ep_CREATININE_std', 'ep_GLU_RAN_std', 'ep_HGB_std', 'ep_RBC_std', 'ep_WBC_std', 
                  'lp_ALT_avg', 'lp_ANC_avg', 'lp_AST_avg', 'lp_BUN_avg', 'lp_CALCIUM_avg', 'lp_CHLORIDE_avg', 
                  'lp_CREATININE_avg', 'lp_GLU_RAN_avg', 'lp_HGB_avg', 'lp_RBC_avg', 'lp_WBC_avg', 
                  'lp_ALT_min', 'lp_ANC_min', 'lp_AST_min', 'lp_BUN_min', 'lp_CALCIUM_min', 'lp_CHLORIDE_min', 
                  'lp_CREATININE_min', 'lp_GLU_RAN_min', 'lp_HGB_min', 'lp_RBC_min', 'lp_WBC_min', 
                  'lp_ALT_max', 'lp_ANC_max', 'lp_AST_max', 'lp_BUN_max', 'lp_CALCIUM_max', 'lp_CHLORIDE_max', 
                  'lp_CREATININE_max', 'lp_GLU_RAN_max', 'lp_HGB_max', 'lp_RBC_max', 'lp_WBC_max', 
                  'lp_ALT_std', 'lp_ANC_std', 'lp_AST_std', 'lp_BUN_std', 'lp_CALCIUM_std', 'lp_CHLORIDE_std', 
                  'lp_CREATININE_std', 'lp_GLU_RAN_std', 'lp_HGB_std', 'lp_RBC_std', 'lp_WBC_std']

print(f"Base features (vitals & labs): {len(feats_to_train)}")

# Medical history features - original set
orig_cos = ['familyhx_dm_ever', 'dep_priorlmp', 'smk_prior', 'alc_prior', 'pcos', 'hxpe_e', 
            'hxghtn', 'hxgdm', 'hxabrt', 'mage', 'PPREG_BMI', 'parity']

# Medical history features - additional comorbidities
new_cos = ['chest_pain', 'uti', 'hyperkeratosis', 'carp_tunnel', 'hypercholes', 'cyst_ovary', 
           'wart', 'pain_rightquad', 'resp_infection', 'pain_neck', 'pain_joint', 'pain_chronic', 
           'acne', 'breast_lump', 'pain_limb', 'ab_pain', 'ten_headache', 'headache', 
           'viraldisease', 'viraldisease_preg']

for i in range(len(orig_cos)):
    feats_to_train.append(orig_cos[i])

for i in range(len(new_cos)):
    feats_to_train.append(new_cos[i])

print(f"After adding medical history: {len(feats_to_train)}")

# Race/ethnicity categories
feats_to_train.append('Hispanic')
feats_to_train.append('NH_White')
feats_to_train.append('NH_Black')
feats_to_train.append('NH_Asian')
feats_to_train.append('Other')

print(f"After adding race/ethnicity: {len(feats_to_train)}")

# Conditional features based on outcome
if outcome != 'Gestational Diabetes_12':
    feats_to_train.append('dm')
    print(f"  Added 'dm' (diabetes) - not predicting GDM")
else:
    print(f"  Excluded 'dm' - predicting GDM")
    
if (outcome != 'Pre-eclampsia_12') and (outcome != 'Gestational_Hypertension_12'):
    feats_to_train.append('chtn_prepreg')
    print(f"  Added 'chtn_prepreg' (chronic hypertension)")
else:
    print(f"  Excluded 'chtn_prepreg' - predicting PE or GHTN")

# For other outcomes (not GDM/PE/GHTN), add additional pregnancy-related features
if (outcome != 'Gestational Diabetes_12') and (outcome != 'Pre-eclampsia_12') and (outcome != 'Gestational_Hypertension_12'):
    feats_to_train.append('GDM_1')
    feats_to_train.append('PE')
    feats_to_train.append('GHTN')
    feats_to_train.append('E')
    feats_to_train.append('preg_single')
    feats_to_train.append('chtn_earlypreg')
    print(f"  Added pregnancy complication features for other outcomes")

print(f"\nFinal feature list length: {len(feats_to_train)}")
'''

'\n# Base features: ALL vitals and labs (pp_, ep_, lp_)\n# Note: lp_ features will be removed for 12-week models, but we include them here\n# so the logic works for models at different time points\n\n\nfeats_to_train = [\'pp_dbp_avg\', \'pp_hr_avg\', \'pp_sbp_avg\', \'pp_dbp_min\', \'pp_hr_min\', \'pp_sbp_min\', \n                  \'pp_dbp_max\', \'pp_hr_max\', \'pp_sbp_max\', \'pp_dbp_std\', \'pp_hr_std\', \'pp_sbp_std\', \n                  \'ep_dbp_avg\', \'ep_hr_avg\', \'ep_sbp_avg\', \'ep_dbp_min\', \'ep_hr_min\', \'ep_sbp_min\', \n                  \'ep_dbp_max\', \'ep_hr_max\', \'ep_sbp_max\', \'ep_dbp_std\', \'ep_hr_std\', \'ep_sbp_std\', \n                  \'lp_dbp_avg\', \'lp_hr_avg\', \'lp_sbp_avg\', \'lp_dbp_min\', \'lp_hr_min\', \'lp_sbp_min\', \n                  \'lp_dbp_max\', \'lp_hr_max\', \'lp_sbp_max\', \'lp_dbp_std\', \'lp_hr_std\', \'lp_sbp_std\', \n                  \'pp_ALT_avg\', \'pp_ANC_avg\', \'pp_AST_avg\', \'pp_BUN_avg\', \'pp_CALCIUM_avg\', \'pp_CHLORID

## Data Filtering Based on Outcome

For certain outcomes, we exclude specific patient groups to avoid confounding:
- **GDM models**: Exclude patients with pre-existing diabetes who don't develop GDM
- **PE/GHTN models**: Exclude patients with chronic hypertension who don't develop PE/GHTN

In [12]:
'''
print(f"\nBefore filtering: {len(df)} pregnancies")

if outcome == 'Gestational Diabetes_12':
    # Exclude patients with pre-existing diabetes who don't develop GDM
    df = df[~((df['dm'] == 1.0) & (df[outcome] == 0))]
    print(f" Excluded patients with pre-existing diabetes and no GDM")
    
if (outcome == 'Pre-eclampsia_12') or (outcome == 'Gestational_Hypertension_12'):
    # Exclude patients with chronic hypertension who don't develop PE/GHTN
    df = df[~((df['chtn_prepreg'] == 1.0) & (df[outcome] == 0))]
    print(f" Excluded patients with chronic hypertension and no {outcome.split('_')[0]}")
    
print(f"After filtering: {len(df)} pregnancies")
'''

'\nprint(f"\nBefore filtering: {len(df)} pregnancies")\n\nif outcome == \'Gestational Diabetes_12\':\n    # Exclude patients with pre-existing diabetes who don\'t develop GDM\n    df = df[~((df[\'dm\'] == 1.0) & (df[outcome] == 0))]\n    print(f" Excluded patients with pre-existing diabetes and no GDM")\n    \nif (outcome == \'Pre-eclampsia_12\') or (outcome == \'Gestational_Hypertension_12\'):\n    # Exclude patients with chronic hypertension who don\'t develop PE/GHTN\n    df = df[~((df[\'chtn_prepreg\'] == 1.0) & (df[outcome] == 0))]\n    print(f" Excluded patients with chronic hypertension and no {outcome.split(\'_\')[0]}")\n    \nprint(f"After filtering: {len(df)} pregnancies")\n'

# Model-Specific Data

Select only the features needed for this model, plus person_id and outcome columns.

In [13]:
'''
# Create model_features list including person_id and outcome
model_features = feats_to_train.copy()
model_features.append('person_id_x')
model_features.append(outcome)

print(f"Model features (including person_id and outcome): {len(model_features)}")

# Select columns that exist in the dataframe
df_external = df[[col for col in model_features if col in df.columns]].copy()

# Add NaN columns for features that don't exist (will be removed later)
for col in model_features:
    if col not in df_external.columns:
        df_external[col] = np.nan 

df = df_external[model_features]

print(f"\nDataset shape: {df.shape}")
print(f"\nPreview:")
print(df.head())

print(f"\nFinal model features list:")
print(model_features)
'''

'\n# Create model_features list including person_id and outcome\nmodel_features = feats_to_train.copy()\nmodel_features.append(\'person_id_x\')\nmodel_features.append(outcome)\n\nprint(f"Model features (including person_id and outcome): {len(model_features)}")\n\n# Select columns that exist in the dataframe\ndf_external = df[[col for col in model_features if col in df.columns]].copy()\n\n# Add NaN columns for features that don\'t exist (will be removed later)\nfor col in model_features:\n    if col not in df_external.columns:\n        df_external[col] = np.nan \n\ndf = df_external[model_features]\n\nprint(f"\nDataset shape: {df.shape}")\nprint(f"\nPreview:")\nprint(df.head())\n\nprint(f"\nFinal model features list:")\nprint(model_features)\n'

# Train-Test Split by Person

We split by person_id, not by pregnancy episode, to prevent data leakage, since some individuals have multiple pregnancies in the cohort. 

In [14]:
'''
# Get unique person IDs
unique_ma = df['person_id_x'].unique()
print(f"\nUnique individuals: {len(unique_ma)}")
print(f"Total pregnancies: {len(df)}")
print(f"Average pregnancies per person: {len(df)/len(unique_ma):.2f}")

# Split person IDs 80-20
train_ma, test_ma = train_test_split(unique_ma, test_size=0.2, random_state=42)

print(f"\nPerson split:")
print(f"  Training persons: {len(train_ma)} ({len(train_ma)/len(unique_ma)*100:.1f}%)")
print(f"  Test persons: {len(test_ma)} ({len(test_ma)/len(unique_ma)*100:.1f}%)")


# Assign all pregnancies from each person to their respective split
train_df = df[df['person_id_x'].isin(train_ma)]
test_df = df[df['person_id_x'].isin(test_ma)]
'''

from sklearn.model_selection import train_test_split

df = pd.read_csv('/data/pregnancy_complications_synthetic.csv')
outcome = 'outcome'
#df['GDM'] = df['outcome']
train_df, test_df = train_test_split(df, test_size=0.20)

X_train = train_df
X_test = test_df

y_train = train_df[outcome]
y_test = test_df[outcome]

print(f"\nPregnancy split:")
print(f"  Training pregnancies: {len(X_train)} ({len(X_train)/len(df)*100:.1f}%)")
print(f"  Test pregnancies: {len(X_test)} ({len(X_test)/len(df)*100:.1f}%)")

print(f"\nOutcome distribution:")
print(f"  Training: {y_train.sum()} positive / {len(y_train)} total ({y_train.sum()/len(y_train)*100:.1f}%)")
print(f"  Test: {y_test.sum()} positive / {len(y_test)} total ({y_test.sum()/len(y_test)*100:.1f}%)")


Pregnancy split:
  Training pregnancies: 4000 (80.0%)
  Test pregnancies: 1000 (20.0%)

Outcome distribution:
  Training: 176 positive / 4000 total (4.4%)
  Test: 46 positive / 1000 total (4.6%)


## Feature Cleaning

Remove columns that:
1. Are all NaN in the training set (no information)
2. Are person_id or outcome (not predictive features)

In [15]:
# Identify columns that are all NaN in training data
nan_columns_train = X_train.columns[X_train.isna().all()]

print(f"\nColumns with all NaN values in training set: {len(nan_columns_train)}")
print(list(nan_columns_train))

# Drop outcome and person_id from feature matrices
X_train = X_train.drop(columns=outcome)
X_test = X_test.drop(columns=outcome)

X_train = X_train.drop(columns='id') #person_id_x' in AoU
X_test = X_test.drop(columns='id')

# Drop all-NaN columns
X_train = X_train.drop(columns=nan_columns_train)
X_test = X_test.drop(columns=nan_columns_train)
df = df.drop(columns=nan_columns_train)

# Update feature list to match what's actually in the data
feats_to_train = list(X_train.columns)

print(f"\n Final feature count: {len(feats_to_train)}")
print(f"\nFinal feature list:")
print(feats_to_train)

print(f"\nFinal shapes:")
print(f"  X_train: {X_train.shape}")
print(f"  X_test: {X_test.shape}")


Columns with all NaN values in training set: 0
[]

 Final feature count: 308

Final feature list:
['Unnamed: 0', 'pp_ALT_avg', 'pp_ALT_min', 'pp_ALT_max', 'pp_ALT_std', 'pp_AST_avg', 'pp_AST_min', 'pp_AST_max', 'pp_AST_std', 'pp_CREATININE_avg', 'pp_CREATININE_min', 'pp_CREATININE_max', 'pp_CREATININE_std', 'pp_GLU_F_avg', 'pp_GLU_F_min', 'pp_GLU_F_max', 'pp_GLU_F_std', 'pp_GLU_RAN_avg', 'pp_GLU_RAN_min', 'pp_GLU_RAN_max', 'pp_GLU_RAN_std', 'pp_HGB_avg', 'pp_HGB_min', 'pp_HGB_max', 'pp_HGB_std', 'pp_RBC_avg', 'pp_RBC_min', 'pp_RBC_max', 'pp_RBC_std', 'pp_WBC_avg', 'pp_WBC_min', 'pp_WBC_max', 'pp_WBC_std', 'pp_ANC_avg', 'pp_ANC_min', 'pp_ANC_max', 'pp_ANC_std', 'pp_CHLORIDE_avg', 'pp_CHLORIDE_min', 'pp_CHLORIDE_max', 'pp_CHLORIDE_std', 'pp_CALCIUM_avg', 'pp_CALCIUM_min', 'pp_CALCIUM_max', 'pp_CALCIUM_std', 'pp_BUN_avg', 'pp_BUN_min', 'pp_BUN_max', 'pp_BUN_std', 'pp_GTT75_2_avg', 'pp_GTT75_2_min', 'pp_GTT75_2_max', 'pp_GTT75_2_std', 'pp_GTT_1_avg', 'pp_GTT_1_min', 'pp_GTT_1_max', 'pp_GT

# Data Preprocessing

In [16]:
# Select only the features we're training on
X_train = X_train[feats_to_train].copy()
scaler = MinMaxScaler()

print(f"\nBefore scaling:")
print(f"  X_train shape: {X_train.shape}")
print(f"  Features: {len(feats_to_train)}")

# Fit scaler on training data and transform both train and test
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test[feats_to_train])

print(f"\nAfter scaling:")
print(f"  X_train shape: {X_train.shape}")
print(f"  X_test shape: {X_test.shape}")



Before scaling:
  X_train shape: (4000, 308)
  Features: 308

After scaling:
  X_train shape: (4000, 308)
  X_test shape: (1000, 308)


# Model Training

In [17]:
print(f"\nTotal final: {X_train.shape[0] + X_test.shape[0]}")
print(f"Total positive cases: {int(len(y_train[y_train == 1]) + len(y_test[y_test == 1]))}")

# XGBoost hyperparameter grid
param_grid = {
    'lambda': [0.5, 1, 1.5],
    'subsample': [0.75, 1],
    'max_depth': [2, 4, 6, 8],
    'eta': [0.01, 0.05, 0.1],
    'gamma': [0, 0.25, 0.5]
}

print(f"\nHyperparameter grid:")
for param, values in param_grid.items():
    print(f"  {param}: {values}")

xgb = XGBClassifier(random_state=42)

print(f"\nPerforming GridSearchCV (5-fold CV)...")
cv_rf = GridSearchCV(
    estimator=xgb, 
    param_grid=param_grid, 
    refit=True, 
    cv=5, 
    scoring='roc_auc'
).fit(X_train, y_train)

print(f"\n Training complete")
print(f"\nBest Parameters: {cv_rf.best_params_}")
print(f"Best CV AUC: {cv_rf.best_score_:.4f}")


Total final: 5000
Total positive cases: 222

Hyperparameter grid:
  lambda: [0.5, 1, 1.5]
  subsample: [0.75, 1]
  max_depth: [2, 4, 6, 8]
  eta: [0.01, 0.05, 0.1]
  gamma: [0, 0.25, 0.5]

Performing GridSearchCV (5-fold CV)...

 Training complete

Best Parameters: {'eta': 0.01, 'gamma': 0, 'lambda': 1, 'max_depth': 2, 'subsample': 0.75}
Best CV AUC: 0.6309


In [18]:
feat_importances = cv_rf.best_estimator_.feature_importances_

# Feature Importance 

In [19]:
# Use sorted() with zip() to pair feature names with importances

sorted_importances = sorted(zip(feats_to_train, feat_importances), key=lambda x: x[1], reverse=True)

# Then iterate through the sorted list
for feature, importance in sorted_importances[:20]:
    print(f"  {feature}: {importance:.4f}")

  pp_GLU_RAN_min: 0.0281
  lp_GTT_2_max: 0.0219
  lp_CALCIUM_avg: 0.0195
  pp_GTTPM_2_avg: 0.0193
  lp_CALCIUM_max: 0.0191
  pp_GLU_F_avg: 0.0177
  lp_ANC_std: 0.0172
  pp_CREATININE_avg: 0.0172
  pp_GLU_F_min: 0.0171
  lp_GTT_2_avg: 0.0171
  lp_dbp_avg: 0.0166
  pp_GLU_RAN_max: 0.0165
  lp_RBC_std: 0.0163
  ep_sbp_max: 0.0158
  pp_ALT_max: 0.0155
  ep_GTT_1_max: 0.0153
  pp_HGB_avg: 0.0150
  pp_BUN_std: 0.0148
  ep_BUN_std: 0.0147
  ep_WBC_COR_max: 0.0144


# Model Evaluation on Test Set

In [20]:
# Predictions
y_pred = cv_rf.predict(X_test)
y_prob = cv_rf.predict_proba(X_test)[:, 1]
y_prob = np.clip(y_prob, 0, 1)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
balanced_acc = balanced_accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

# PR AUC
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_prob)
pr_auc = auc(recall_curve, precision_curve)

brier = brier_score_loss(y_test, y_prob)
logloss = log_loss(y_test, y_prob)

prob_metrics_1 = {
    'accuracy': accuracy,
    'balanced_accuracy': balanced_acc,
    'precision': precision,
    'recall': recall,
    'roc_auc': roc_auc,
    'pr_auc': pr_auc,
    'brier_score': brier,
    'log_loss': logloss
}

print(f"\nPerformance Metrics:")
print(f"  ROC AUC:            {roc_auc:.4f}")
print(f"  PR AUC:             {pr_auc:.4f}")
print(f"  Accuracy:           {accuracy:.4f}")
print(f"  Balanced Accuracy:  {balanced_acc:.4f}")
print(f"  Precision:          {precision:.4f}")
print(f"  Recall:             {recall:.4f}")
print(f"  Brier Score:        {brier:.4f}")
print(f"  Log Loss:           {logloss:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion Matrix:")
print(f"  TN: {cm[0,0]}, FP: {cm[0,1]}")
print(f"  FN: {cm[1,0]}, TP: {cm[1,1]}")


Performance Metrics:
  ROC AUC:            0.7016
  PR AUC:             0.1241
  Accuracy:           0.9540
  Balanced Accuracy:  0.5000
  Precision:          0.0000
  Recall:             0.0000
  Brier Score:        0.0431
  Log Loss:           0.1788

Confusion Matrix:
  TN: 954, FP: 0
  FN: 46, TP: 0


/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [21]:
from sklearn.utils import resample
import numpy as np

# Initialize lists for bootstrap results
aucs = []
pr_aucs = []

n_bootstrap = 500
print(f"Running {n_bootstrap} bootstrap iterations...")

for i in range(n_bootstrap):
    if (i + 1) % 100 == 0:
        print(f"  Progress: {i + 1}/{n_bootstrap}")
    
    # Resample with replacement
    X_test_boot, y_test_boot = resample(X_test, y_test, replace=True, random_state=i)
    
    # Skip if no positive cases in bootstrap sample
    if y_test_boot.sum() == 0:
        continue
    
    # Predictions
    y_prob_boot = cv_rf.predict_proba(X_test_boot)[:, 1]
    y_prob_boot = np.clip(y_prob_boot, 0, 1)
    
    # Calculate metrics
    try:
        roc_auc_boot = roc_auc_score(y_test_boot, y_prob_boot)
        precision_curve, recall_curve, _ = precision_recall_curve(y_test_boot, y_prob_boot)
        pr_auc_boot = auc(recall_curve, precision_curve)
        
        aucs.append(roc_auc_boot)
        pr_aucs.append(pr_auc_boot)
    except:
        continue

# Calculate 95% confidence intervals
roc_ci_low, roc_ci_high = np.percentile(aucs, [2.5, 97.5])
pr_ci_low, pr_ci_high = np.percentile(pr_aucs, [2.5, 97.5])

print(f"\n======================================================================")
print(f"BOOTSTRAP CONFIDENCE INTERVALS (95%)")
print(f"======================================================================")
print(f"ROC AUC: {roc_auc:.3f} [{roc_ci_low:.3f}, {roc_ci_high:.3f}]")
print(f"PR AUC:  {pr_auc:.3f} [{pr_ci_low:.3f}, {pr_ci_high:.3f}]")


Running 500 bootstrap iterations...
  Progress: 100/500
  Progress: 200/500
  Progress: 300/500
  Progress: 400/500
  Progress: 500/500

BOOTSTRAP CONFIDENCE INTERVALS (95%)
ROC AUC: 0.702 [0.612, 0.785]
PR AUC:  0.124 [0.071, 0.198]


# Save Model

In [22]:
'''
# filename based on outcome
outcome_short = outcome.split('_')[0]  # e.g., 'Pre-eclampsia' -> 'Pre-eclampsia', then take abbreviation
outcome_abbrev = {'Pre-eclampsia': 'PE', 'Gestational': 'GHTN', 'Gestational Diabetes': 'GDM'}.get(outcome_short, outcome_short)
model_filename = f'AoU_Internal_{outcome_abbrev}.pkl'

# Save the trained model
with open(model_filename, 'wb') as f:
    pickle.dump(cv_rf, f)
print(f" Model saved: {model_filename}")

results = {
    'model': cv_rf,
    'features': feats_to_train,
    'scaler': scaler,
    'performance': prob_metrics_1
}

full_filename = f'AoU_Internal_{outcome_abbrev}_full.pkl'
with open(full_filename, 'wb') as f:
    pickle.dump(results, f)
print(f" results saved: {full_filename}")
'''

'\n# filename based on outcome\noutcome_short = outcome.split(\'_\')[0]  # e.g., \'Pre-eclampsia\' -> \'Pre-eclampsia\', then take abbreviation\noutcome_abbrev = {\'Pre-eclampsia\': \'PE\', \'Gestational\': \'GHTN\', \'Gestational Diabetes\': \'GDM\'}.get(outcome_short, outcome_short)\nmodel_filename = f\'AoU_Internal_{outcome_abbrev}.pkl\'\n\n# Save the trained model\nwith open(model_filename, \'wb\') as f:\n    pickle.dump(cv_rf, f)\nprint(f" Model saved: {model_filename}")\n\nresults = {\n    \'model\': cv_rf,\n    \'features\': feats_to_train,\n    \'scaler\': scaler,\n    \'performance\': prob_metrics_1\n}\n\nfull_filename = f\'AoU_Internal_{outcome_abbrev}_full.pkl\'\nwith open(full_filename, \'wb\') as f:\n    pickle.dump(results, f)\nprint(f" results saved: {full_filename}")\n'

# Per-State Evaluation

## Add State Information

In [23]:
'''
# Need to reload the full df to get person_id_x back
df = df_external[model_features]

# Query to get state of residence
state_query = f"""
SELECT 
    person_id,
    COALESCE(state_of_residence_concept_id, 0) as state_concept_id,
    state.concept_name as state_name
FROM `{version}.person` person
LEFT JOIN `{version}.concept` state
    ON person.state_of_residence_concept_id = state.concept_id
WHERE person_id IN ({','.join(map(str, df['person_id_x'].unique()))})
"""

df_states = pd.read_gbq(state_query, dialect='standard')

# Merge with main dataframe
df_states = df_states.rename(columns={'person_id': 'person_id_for_merge'})
df = df.merge(df_states, 
              left_on='person_id_x', 
              right_on='person_id_for_merge', 
              how='left')
df = df.drop(columns=['person_id_for_merge'])

# Clean state names (remove 'PII State: ' prefix)
df['state_clean'] = df['state_name'].str.replace('PII State: ', '', regex=False)

print(f" States added: {df['state_clean'].nunique()} unique states")
print(f"\nTop 10 states by sample size:")
print(df['state_clean'].value_counts().head(10))
'''

'\n# Need to reload the full df to get person_id_x back\ndf = df_external[model_features]\n\n# Query to get state of residence\nstate_query = f"""\nSELECT \n    person_id,\n    COALESCE(state_of_residence_concept_id, 0) as state_concept_id,\n    state.concept_name as state_name\nFROM `{version}.person` person\nLEFT JOIN `{version}.concept` state\n    ON person.state_of_residence_concept_id = state.concept_id\nWHERE person_id IN ({\',\'.join(map(str, df[\'person_id_x\'].unique()))})\n"""\n\ndf_states = pd.read_gbq(state_query, dialect=\'standard\')\n\n# Merge with main dataframe\ndf_states = df_states.rename(columns={\'person_id\': \'person_id_for_merge\'})\ndf = df.merge(df_states, \n              left_on=\'person_id_x\', \n              right_on=\'person_id_for_merge\', \n              how=\'left\')\ndf = df.drop(columns=[\'person_id_for_merge\'])\n\n# Clean state names (remove \'PII State: \' prefix)\ndf[\'state_clean\'] = df[\'state_name\'].str.replace(\'PII State: \', \'\', regex=F

## Evaluation

In [24]:
'''
print(f"PER-STATE ANALYSIS: AoU INTERNAL {outcome_abbrev} MODEL")

# Identify which rows from df correspond to the test set
print(f"\nMatching test set to state information...")
print(f"  X_test type: {type(X_test)}")
print(f"  y_test type: {type(y_test)}")

# Use y_test index to identify test set rows
if isinstance(y_test, pd.Series):
    test_indices = y_test.index
    print(f"  Using y_test.index to identify test set rows")
else:
    print(f"  WARNING: y_test is not a Series, cannot track indices properly")

# Get state information for test set
df_test = df.loc[df.index.isin(test_indices)]

print(f"\n  Test set size: {len(y_test)}")
print(f"  Test set with state info: {len(df_test)}")
print(f"  Coverage: {len(df_test)/len(y_test)*100:.1f}%")

# Get states with sufficient sample size (n >= 50)
states = df_test['state_clean'].value_counts()
print(f"\nState distribution in test set:")
print(states)

states_to_analyze = states[states >= 50].index.tolist()
print(f"\n Analyzing {len(states_to_analyze)} states with n≥50")
'''

'\nprint(f"PER-STATE ANALYSIS: AoU INTERNAL {outcome_abbrev} MODEL")\n\n# Identify which rows from df correspond to the test set\nprint(f"\nMatching test set to state information...")\nprint(f"  X_test type: {type(X_test)}")\nprint(f"  y_test type: {type(y_test)}")\n\n# Use y_test index to identify test set rows\nif isinstance(y_test, pd.Series):\n    test_indices = y_test.index\n    print(f"  Using y_test.index to identify test set rows")\nelse:\n    print(f"  WARNING: y_test is not a Series, cannot track indices properly")\n\n# Get state information for test set\ndf_test = df.loc[df.index.isin(test_indices)]\n\nprint(f"\n  Test set size: {len(y_test)}")\nprint(f"  Test set with state info: {len(df_test)}")\nprint(f"  Coverage: {len(df_test)/len(y_test)*100:.1f}%")\n\n# Get states with sufficient sample size (n >= 50)\nstates = df_test[\'state_clean\'].value_counts()\nprint(f"\nState distribution in test set:")\nprint(states)\n\nstates_to_analyze = states[states >= 50].index.tolist()\

In [25]:
'''
def evaluate_state_internal(state_name, df_test_subset, y_test_series, X_test_array, model, min_cases=10):
    """
    Evaluate internal model performance for a specific state
    
    Parameters:
    - state_name: Name of state to evaluate
    - df_test_subset: Test set dataframe with state information
    - y_test_series: True labels (pandas Series with index)
    - X_test_array: Test features (numpy array or DataFrame)
    - model: Trained model
    - min_cases: Minimum number of positive cases required
    
    Returns:
    - Dictionary with performance metrics or None if insufficient data
    """
    # Get indices for this state
    state_mask = df_test_subset['state_clean'] == state_name
    state_df = df_test_subset[state_mask]
    
    if len(state_df) == 0:
        return None
    
    # Get corresponding positions in test arrays
    matching_indices = state_df.index.intersection(y_test_series.index)
    
    if len(matching_indices) == 0:
        return None
    
    # Get y values for this state
    y_state = y_test_series.loc[matching_indices]
    
    # Get X values - need to find positions in the array
    if isinstance(X_test_array, np.ndarray):
        positions = [y_test_series.index.get_loc(idx) for idx in matching_indices]
        X_state = X_test_array[positions]
    else:
        X_state = X_test_array.loc[matching_indices]
    
    n_positive = y_state.sum()
    n_total = len(y_state)
    
    # Skip if insufficient data
    if n_positive < min_cases or n_total < 50:
        return None
    
    # Generate predictions
    probs = model.predict_proba(X_state)[:,1]
    probs = np.clip(probs, 0, 1)
    
    # Calculate metrics
    roc_auc = roc_auc_score(y_state, probs)
    precision, recall, _ = precision_recall_curve(y_state, probs)
    pr_auc = auc(recall, precision)
    
    # Bootstrap confidence intervals (500 iterations)
    aucs = []
    for _ in range(500):
        indices_boot = np.random.choice(len(y_state), size=len(y_state), replace=True)
        if isinstance(X_state, np.ndarray):
            X_boot = X_state[indices_boot]
        else:
            X_boot = X_state.iloc[indices_boot]
        y_boot = y_state.iloc[indices_boot] if isinstance(y_state, pd.Series) else y_state[indices_boot]
        
        probs_boot = model.predict_proba(X_boot)[:,1]
        probs_boot = np.clip(probs_boot, 0, 1)
        try:
            aucs.append(roc_auc_score(y_boot, probs_boot))
        except:
            continue
    
    ci_lower, ci_upper = np.percentile(aucs, [2.5, 97.5])
    
    return {
        'state': state_name,
        'n_total': n_total,
        'n_positive': n_positive,
        'prevalence': n_positive / n_total * 100,
        'roc_auc': roc_auc,
        'pr_auc': pr_auc,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
        'ci_width': ci_upper - ci_lower
    }

'''

'\ndef evaluate_state_internal(state_name, df_test_subset, y_test_series, X_test_array, model, min_cases=10):\n    """\n    Evaluate internal model performance for a specific state\n    \n    Parameters:\n    - state_name: Name of state to evaluate\n    - df_test_subset: Test set dataframe with state information\n    - y_test_series: True labels (pandas Series with index)\n    - X_test_array: Test features (numpy array or DataFrame)\n    - model: Trained model\n    - min_cases: Minimum number of positive cases required\n    \n    Returns:\n    - Dictionary with performance metrics or None if insufficient data\n    """\n    # Get indices for this state\n    state_mask = df_test_subset[\'state_clean\'] == state_name\n    state_df = df_test_subset[state_mask]\n    \n    if len(state_df) == 0:\n        return None\n    \n    # Get corresponding positions in test arrays\n    matching_indices = state_df.index.intersection(y_test_series.index)\n    \n    if len(matching_indices) == 0:\n      

In [26]:
'''
# Evaluate each state
print("\n" + "-"*70)
internal_results = []
for state in states_to_analyze:
    result = evaluate_state_internal(state, df_test, y_test, X_test, cv_rf)
    if result is not None:
        internal_results.append(result)
        print(f"  {state}: AUC={result['roc_auc']:.3f} ({result['n_positive']}/{result['n_total']})")
'''

'\n# Evaluate each state\nprint("\n" + "-"*70)\ninternal_results = []\nfor state in states_to_analyze:\n    result = evaluate_state_internal(state, df_test, y_test, X_test, cv_rf)\n    if result is not None:\n        internal_results.append(result)\n        print(f"  {state}: AUC={result[\'roc_auc\']:.3f} ({result[\'n_positive\']}/{result[\'n_total\']})")\n'

In [27]:
'''
# Summarize results
if len(internal_results) == 0:
    print("\n  No states had sufficient data for analysis")
else:
    internal_results_df = pd.DataFrame(internal_results)
    internal_results_df = internal_results_df.sort_values('roc_auc', ascending=False)
    
    print(f"\n Completed {len(internal_results_df)} states")
    print(f"AUC range: [{internal_results_df['roc_auc'].min():.3f}, {internal_results_df['roc_auc'].max():.3f}]")
    print(f"Mean AUC: {internal_results_df['roc_auc'].mean():.3f}")
    print(f"Std AUC: {internal_results_df['roc_auc'].std():.3f}")
    
    # Save results
    results_filename = f'AoU_Internal_{outcome_abbrev}_state_results.csv'
    internal_results_df.to_csv(results_filename, index=False)
    print(f"\n Results saved to: {results_filename}")
    
    # Display results table
    print("\ results by state:")
    print(internal_results_df.to_string(index=False))
    '''

<>:1: SyntaxWarning: invalid escape sequence '\ '
<>:1: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_158/2721387668.py:1: SyntaxWarning: invalid escape sequence '\ '
  '''


'\n# Summarize results\nif len(internal_results) == 0:\n    print("\n  No states had sufficient data for analysis")\nelse:\n    internal_results_df = pd.DataFrame(internal_results)\n    internal_results_df = internal_results_df.sort_values(\'roc_auc\', ascending=False)\n    \n    print(f"\n Completed {len(internal_results_df)} states")\n    print(f"AUC range: [{internal_results_df[\'roc_auc\'].min():.3f}, {internal_results_df[\'roc_auc\'].max():.3f}]")\n    print(f"Mean AUC: {internal_results_df[\'roc_auc\'].mean():.3f}")\n    print(f"Std AUC: {internal_results_df[\'roc_auc\'].std():.3f}")\n    \n    # Save results\n    results_filename = f\'AoU_Internal_{outcome_abbrev}_state_results.csv\'\n    internal_results_df.to_csv(results_filename, index=False)\n    print(f"\n Results saved to: {results_filename}")\n    \n    # Display results table\n    print("\\ results by state:")\n    print(internal_results_df.to_string(index=False))\n    '

## Visualizations

In [28]:
'''
if len(internal_results) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(f'AoU Internal Model ({outcome_abbrev}) - Per-State Performance Analysis', 
                 fontsize=16, fontweight='bold', y=0.995)
    
    # Plot 1: State ranking by AUC
    ax1 = axes[0, 0]
    results_sorted = internal_results_df.sort_values('roc_auc', ascending=True)
    colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(results_sorted)))
    bars = ax1.barh(range(len(results_sorted)), results_sorted['roc_auc'],
                    color=colors, edgecolor='black', linewidth=0.5)
    
    ax1.set_yticks(range(len(results_sorted)))
    ax1.set_yticklabels(results_sorted['state'], fontsize=9)
    mean_auc = internal_results_df['roc_auc'].mean()
    ax1.axvline(x=mean_auc, color='blue', linestyle='--', alpha=0.6, linewidth=2,
               label=f'Mean AUC={mean_auc:.3f}')
    ax1.set_xlabel('ROC AUC', fontsize=12, fontweight='bold')
    ax1.set_title('State Rankings by AUC', fontsize=13, fontweight='bold')
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3, axis='x')
    
    # Plot 2: Sample sizes and prevalence
    ax2 = axes[0, 1]
    ax2_twin = ax2.twinx()
    x_pos = range(len(results_sorted))
    
    ax2.bar(x_pos, results_sorted['n_total'], alpha=0.6, 
           color='steelblue', edgecolor='black', linewidth=0.5, label='Sample Size')
    ax2_twin.plot(x_pos, results_sorted['prevalence'], 'ro-', linewidth=2.5, 
                 markersize=8, label=f'{outcome_abbrev} Rate (%)')
    
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(results_sorted['state'], rotation=45, ha='right', fontsize=8)
    ax2.set_ylabel('Sample Size', fontsize=11, fontweight='bold', color='steelblue')
    ax2_twin.set_ylabel(f'{outcome_abbrev} Rate (%)', fontsize=11, fontweight='bold', color='darkred')
    ax2.set_title('Sample Characteristics by State', fontsize=13, fontweight='bold')
    ax2.legend(loc='upper left', fontsize=9)
    ax2_twin.legend(loc='upper right', fontsize=9)
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Plot 3: Confidence interval widths vs sample size
    ax3 = axes[1, 0]
    ax3.scatter(internal_results_df['n_total'], internal_results_df['ci_width'], 
               s=100, alpha=0.6, edgecolors='black')
    for idx, row in internal_results_df.iterrows():
        ax3.annotate(row['state'], (row['n_total'], row['ci_width']), 
                    fontsize=8, alpha=0.7, xytext=(5, 5), textcoords='offset points')
    ax3.set_xlabel('Sample Size', fontsize=12, fontweight='bold')
    ax3.set_ylabel('CI Width (95%)', fontsize=12, fontweight='bold')
    ax3.set_title('Uncertainty vs Sample Size', fontsize=13, fontweight='bold')
    ax3.grid(True, alpha=0.3)
    
    # Plot 4: AUC distribution
    ax4 = axes[1, 1]
    ax4.hist(internal_results_df['roc_auc'], bins=10, edgecolor='black', 
            color='skyblue', alpha=0.7)
    ax4.axvline(x=mean_auc, color='red', linestyle='--', linewidth=2, 
               label=f'Mean={mean_auc:.3f}')
    ax4.set_xlabel('ROC AUC', fontsize=12, fontweight='bold')
    ax4.set_ylabel('Number of States', fontsize=12, fontweight='bold')
    ax4.set_title('Distribution of State Performance', fontsize=13, fontweight='bold')
    ax4.legend(fontsize=10)
    ax4.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    viz_filename = f'AoU_Internal_{outcome_abbrev}_state_analysis.png'
    plt.savefig(viz_filename, dpi=300, bbox_inches='tight')
    print(f" Visualization saved: {viz_filename}")
    plt.show()
    '''

'\nif len(internal_results) > 0:\n    fig, axes = plt.subplots(2, 2, figsize=(16, 12))\n    fig.suptitle(f\'AoU Internal Model ({outcome_abbrev}) - Per-State Performance Analysis\', \n                 fontsize=16, fontweight=\'bold\', y=0.995)\n    \n    # Plot 1: State ranking by AUC\n    ax1 = axes[0, 0]\n    results_sorted = internal_results_df.sort_values(\'roc_auc\', ascending=True)\n    colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(results_sorted)))\n    bars = ax1.barh(range(len(results_sorted)), results_sorted[\'roc_auc\'],\n                    color=colors, edgecolor=\'black\', linewidth=0.5)\n    \n    ax1.set_yticks(range(len(results_sorted)))\n    ax1.set_yticklabels(results_sorted[\'state\'], fontsize=9)\n    mean_auc = internal_results_df[\'roc_auc\'].mean()\n    ax1.axvline(x=mean_auc, color=\'blue\', linestyle=\'--\', alpha=0.6, linewidth=2,\n               label=f\'Mean AUC={mean_auc:.3f}\')\n    ax1.set_xlabel(\'ROC AUC\', fontsize=12, fontweight=\'bold\')\n    ax1